In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler

# Carregar dados
df_historico = pd.read_csv("arquivos/historico.csv", dtype={
    "userId": "string", "page": "string", "timestampHistory": "string",
    "numberOfClicks": "int", "timeOnPage": "int",
    "scrollPercentage": "float", "pageVisitsCount": "int"
})
df_noticias = pd.read_csv("arquivos/noticias_clusterizadas.csv", dtype={"page": "string", "cluster": "int", "score": "float"})

# Carregar modelo e scaler
with open("pkls/modelo_final.pkl", "rb") as f:
    modelo = pickle.load(f)

with open("pkls/scaler_final.pkl", "rb") as f:
    scaler = pickle.load(f)

In [2]:
# Top 10 notícias gerais (melhor score)
top_10 = df_noticias.sort_values("score", ascending=False).head(10)

# Salvar Top 10 notícias gerais
top_10.to_csv("arquivos/top10_noticias.csv", index=False)

# Mostrar top 10
display(top_10)

,page,title,text,url,issued,cluster,score,date_ranking
23198,d2593c3d-2347-40d9-948c-b6065e8459a9,Anestesista é preso em flagrante por estupro d...,Anestesista é preso em flagrante por estupro d...,http://g1.globo.com/rj/rio-de-janeiro/noticia/...,2022-07-11 03:20:57,2,0.796550,26720
71660,f6b5d170-48b9-4f8e-88d4-c84b6668f3bd,Diretor da Caixa Econômica Federal é encontrad...,Diretor da Caixa Econômica Federal é encontrad...,http://g1.globo.com/politica/blog/andreia-sadi...,2022-07-20 11:14:32,5,0.736898,19470
60121,1f32787b-de2b-49be-8c20-ddaeae34cc22,Filha é presa por golpe estimado em R$ 725 mil...,Filha é presa por golpe estimado em R$ 725 mil...,http://g1.globo.com/rj/rio-de-janeiro/noticia/...,2022-08-10 09:55:29,3,0.699337,3443
15859,f0a78e58-ec7e-494c-9462-fbd6446a9a89,Caso Bárbara: suspeito de envolvimento no assa...,Caso Bárbara: suspeito de envolvimento no assa...,http://g1.globo.com/mg/minas-gerais/noticia/20...,2022-08-03 20:08:46,2,0.676280,8306
26314,6a83890a-d9e9-4f6b-a6c6-90d031785bbf,Pizzaria recebe PIX falso e entrega refrigeran...,Pizzaria recebe PIX falso e entrega refrigeran...,http://g1.globo.com/pi/piaui/noticia/2022/07/2...,2022-07-27 13:54:29,7,0.653750,14056
66042,bf257382-74fb-4392-ad6a-143240e39f81,"Jô Soares, ícone do humor e da TV, morre em Sã...","Jô Soares, ícone do humor e da TV, morre em Sã...",http://g1.globo.com/sp/sao-paulo/noticia/2022/...,2022-08-05 08:23:07,7,0.636186,7107
223980,855d20b7-53f2-4678-a10f-55402d085018,‘Tímido e discreto’: saiba quem era filho de C...,‘Tímido e discreto’: saiba quem era filho de C...,http://g1.globo.com/go/goias/noticia/2022/07/0...,2022-07-04 08:54:20,5,0.633433,31932
120348,1c27cf97-b20c-4e40-b1f1-288b721517b3,Vídeo flagra homem atirando na cabeça de vizin...,Vídeo flagra homem atirando na cabeça de vizin...,http://g1.globo.com/ms/mato-grosso-do-sul/noti...,2022-07-25 15:22:38,2,0.625985,15746
116040,a36c98b5-f159-48f8-9f5a-1fc6ea9956c8,"Campeão mundial de jiu-jítsu, Leandro Lo é bal...","Campeão mundial de jiu-jítsu, Leandro Lo é bal...",http://g1.globo.com/sp/sao-paulo/noticia/2022/...,2022-08-07 12:37:58,2,0.625249,5639
164776,4c63d7cd-4902-4ffb-9b94-578b1b2151f0,Vídeos mostram diferentes ângulos do ataque ao...,Vídeos mostram diferentes ângulos do ataque ao...,http://g1.globo.com/mundo/noticia/2022/07/08/m...,2022-07-08 10:12:06,2,0.621897,28381


In [3]:
# Processar histórico de leitura
df_historico["timestampHistory"] = pd.to_datetime(df_historico["timestampHistory"], errors="coerce")

# Associar cluster ao histórico
df_historico = df_historico.merge(df_noticias[["page", "cluster"]], left_on="page", right_on="page", how="left")

# Ordenar por usuário e timestamp
df_historico = df_historico.sort_values(["userId", "timestampHistory"])

# Criar prev_cluster_1, prev_cluster_2, prev_cluster_3 (alinhado ao treino)
df_historico["prev_cluster_1"] = df_historico.groupby("userId")["cluster"].shift(1).fillna(-1).astype(int)
df_historico["prev_cluster_2"] = df_historico.groupby("userId")["cluster"].shift(2).fillna(-1).astype(int)
df_historico["prev_cluster_3"] = df_historico.groupby("userId")["cluster"].shift(3).fillna(-1).astype(int)

# Criar df_user_last com último cluster e os 3 anteriores (recente, penúltimo, antepenúltimo)
df_user_last = df_historico.groupby("userId").tail(1)[[
    "userId", "cluster", "prev_cluster_1", "prev_cluster_2", "prev_cluster_3"
]]
df_user_last = df_user_last.rename(columns={"cluster": "last_cluster"})

# Contagem de visitas por cluster
cluster_ids = df_historico["cluster"].dropna().unique()
df_cluster_count = df_historico.groupby(["userId", "cluster"]).size().unstack(fill_value=0).reset_index()
df_cluster_count.columns = ["userId"] + [f"visitas_cluster_{int(c)}" for c in cluster_ids]

# Tempo e scroll médio por cluster
df_cluster_metrics = df_historico.groupby(["userId", "cluster"]).agg(
    tempo_medio=("timeOnPage", "mean"),
    scroll_medio=("scrollPercentage", "mean")
).unstack(fill_value=0).reset_index()

df_cluster_metrics.columns = ["userId"] + [
    f"{col}_{int(cluster_id)}" for col, cluster_id in df_cluster_metrics.columns[1:]
]

# Consolidar features do usuário
df_user_features = (
    df_user_last
    .merge(df_cluster_count, on="userId", how="left").fillna(0)
    .merge(df_cluster_metrics, on="userId", how="left").fillna(0)
)

# Garantir que contagens são inteiros
for col in df_user_features.columns:
    if col.startswith("visitas_cluster_"):
        df_user_features[col] = df_user_features[col].astype(int)

# Selecionar features para previsão - garantir alinhamento com treinamento
features_treinadas = scaler.feature_names_in_  # Obtém a lista exata de colunas usadas no treinamento

# Reorganizar colunas e preencher colunas faltantes com 0
X = df_user_features.reindex(columns=features_treinadas, fill_value=0)


In [4]:
# Normalizar
X_scaled = scaler.transform(X)

# Prever próximo cluster
df_user_features["proximo_cluster_previsto"] = modelo.predict(X_scaled)

c:\Projetos\project_globo\modelo\venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [5]:
previstos = df_user_features['proximo_cluster_previsto'].nunique()
print(previstos)

8


In [6]:
# Criar lista com as top 10 notícias gerais
top_10_geral_list = top_10["page"].tolist()
print("\nTop 10 geral:")
print(top_10_geral_list)

# Gerar dicionário com top 3 notícias de cada cluster (ordenado por score)
top3_por_cluster = (
    df_noticias
    .sort_values(by="score", ascending=False)
    .groupby("cluster")
    .head(3)  # Pega só top 3 de cada cluster
    .groupby("cluster")["page"]
    .apply(list)  # Transforma em lista
    .to_dict()
)
print("\nTop 3 por cluster:")
for cluster, pages in top3_por_cluster.items():
    print(f"Cluster {cluster}: {pages}")



Top 10 geral:
['d2593c3d-2347-40d9-948c-b6065e8459a9', 'f6b5d170-48b9-4f8e-88d4-c84b6668f3bd', '1f32787b-de2b-49be-8c20-ddaeae34cc22', 'f0a78e58-ec7e-494c-9462-fbd6446a9a89', '6a83890a-d9e9-4f6b-a6c6-90d031785bbf', 'bf257382-74fb-4392-ad6a-143240e39f81', '855d20b7-53f2-4678-a10f-55402d085018', '1c27cf97-b20c-4e40-b1f1-288b721517b3', 'a36c98b5-f159-48f8-9f5a-1fc6ea9956c8', '4c63d7cd-4902-4ffb-9b94-578b1b2151f0']

Top 3 por cluster:
Cluster 0: ['a6e7224d-da3e-468f-bc51-26331659e06a', '21532265-cfd0-4301-a58d-dd615d0189d6', 'fd3c2179-09b0-4b2c-96a9-2d8822f28851']
Cluster 1: ['e628837a-cd62-42e2-8120-35f8cef06eb8', '593dc77c-a112-4a6c-9a0c-88949bd1add0', '3b4892f6-2402-446f-a3d5-ef53c437c9e5']
Cluster 2: ['d2593c3d-2347-40d9-948c-b6065e8459a9', 'f0a78e58-ec7e-494c-9462-fbd6446a9a89', '1c27cf97-b20c-4e40-b1f1-288b721517b3']
Cluster 3: ['1f32787b-de2b-49be-8c20-ddaeae34cc22', 'b79a9385-ab79-4949-a953-2320bfb28bc0', 'ad2c51f7-26ff-4f7e-98c8-32462c62874e']
Cluster 4: ['eebd0c96-3b69-4246-b79a

In [7]:
# Contagem de usuários únicos no histórico completo
total_usuarios_historico = df_historico['userId'].nunique()

# Contar usuários curtos (<= 2 notícias)
usuarios_historico_curto = df_user_features[df_user_features["userId"].isin(
    df_historico["userId"].value_counts()[lambda x: x <= 2].index
)]

total_usuarios_curto = usuarios_historico_curto['userId'].nunique()

# Contar usuários longos (todos que não são curtos)
usuarios_historico_longo = df_user_features[~df_user_features["userId"].isin(usuarios_historico_curto["userId"])]
total_usuarios_longo = usuarios_historico_longo['userId'].nunique()

# Soma dos dois
soma_curtos_longos = total_usuarios_curto + total_usuarios_longo

# Exibir resumo
print(f"Total de usuários no histórico: {total_usuarios_historico}")
print(f"Usuários curtos (<= 2 notícias): {total_usuarios_curto}")
print(f"Usuários longos (> 2 notícias): {total_usuarios_longo}")
print(f"Soma curtos + longos: {soma_curtos_longos}")

# Checar se soma bate com total do histórico 
if soma_curtos_longos == total_usuarios_historico:
    print("\n Soma de curtos e longos bate com o total de usuários no histórico.")
else:
    print("\n Atenção: Soma de curtos e longos NÃO bate com o total de usuários no histórico.")
    print(f" Diferença: {total_usuarios_historico - soma_curtos_longos}")


Total de usuários no histórico: 577942
Usuários curtos (<= 2 notícias): 342752
Usuários longos (> 2 notícias): 235190
Soma curtos + longos: 577942

 Soma de curtos e longos bate com o total de usuários no histórico.


In [8]:
df_curto_cluster = []  
df_curto_top10_final = []  # Armazena todas recomendações de todos os usuários

for row in usuarios_historico_curto.itertuples():
    user_id = row.userId
    last_cluster = row.last_cluster

    noticias_usadas = set()

    # Criar recomendação do cluster
    top1_cluster = top3_por_cluster.get(last_cluster, [])[:1]

    for rn, page in enumerate(top1_cluster, start=1):
        df_curto_cluster.append([user_id, page, rn])  # Sem erro agora
        noticias_usadas.add(page)

    # Criar lista específica para o top 10 geral deste user
    df_curto_top10 = []

    rn = len(top1_cluster) + 1  # Continua numerando

    for page in top_10_geral_list:
        if page not in noticias_usadas:
            df_curto_top10.append([user_id, page, rn])
            noticias_usadas.add(page)
            rn += 1

        if len(df_curto_top10) >= 4:
            break

    # Ao final de cada user, joga o df_curto_top10 dele para o resultado final
    df_curto_top10_final.extend(df_curto_top10)

# Criar DataFrame único com todos os usuários curtos
df_curto_cluster = pd.DataFrame(df_curto_cluster, columns=["userId", "page", "rn"])
df_curto_top10_final = pd.DataFrame(df_curto_top10_final, columns=["userId", "page", "rn"])

# Concatenar cluster + top10
df_recomendacoes_curto = pd.concat([df_curto_cluster, df_curto_top10_final])

print(df_recomendacoes_curto)

# Validação
total_users_curto = df_recomendacoes_curto['userId'].nunique()
total_registros_curto = len(df_recomendacoes_curto)
total_esperado = total_users_curto * 5

print(f"\nQuantidade de usuários curtos: {total_users_curto}")
print(f"Quantidade de registros (recomendações): {total_registros_curto}")
print(f"Quantidade esperada (usuários * 5): {total_esperado}")

if total_registros_curto == total_esperado:
    print("Cada usuário curto recebeu exatamente 5 recomendações.")
else:
    print("Problema: Alguns usuários curtos receberam menos de 5 recomendações.")


                                                    userId  \
0        00007a4e5949a3dba7c977503c53e0873643fe17d0802a...   
1        000087b05ccb95dec5d55e968764285c5403747fc35da2...   
2        00011b1ced626112372206634e0e9b5ccb432da916e83f...   
3        00019abf778947398b46310c3947cc0260f30e79683ae9...   
4        0001b40676c18a37bf25f0b1921ad12513c434cb57db18...   
...                                                    ...   
1371003  ffffa6346e5c3c5219194638c9597f81a08a33e1ffe83f...   
1371004  ffffec1d6956710feb0bb6b05e83849fca3cced6286789...   
1371005  ffffec1d6956710feb0bb6b05e83849fca3cced6286789...   
1371006  ffffec1d6956710feb0bb6b05e83849fca3cced6286789...   
1371007  ffffec1d6956710feb0bb6b05e83849fca3cced6286789...   

                                         page  rn  
0        6a83890a-d9e9-4f6b-a6c6-90d031785bbf   1  
1        6a83890a-d9e9-4f6b-a6c6-90d031785bbf   1  
2        6a83890a-d9e9-4f6b-a6c6-90d031785bbf   1  
3        6a83890a-d9e9-4f6b-a6c6-90d031785bbf  

In [9]:
# Criar lista para armazenar recomendações do cluster e do top10 geral
df_longo_cluster = []
df_longo_top10_final = []

for row in usuarios_historico_longo.itertuples():
    user_id = row.userId
    next_cluster = row.proximo_cluster_previsto

    noticias_usadas = set()

    # Criar recomendação do cluster (Top 3)
    top3_cluster = top3_por_cluster.get(next_cluster, [])[:3]

    for rn, page in enumerate(top3_cluster, start=1):
        df_longo_cluster.append([user_id, page, rn])
        noticias_usadas.add(page)

    # Criar lista específica para o top 10 geral deste user
    df_longo_top10 = []

    rn = len(top3_cluster) + 1  # Começa no 4

    for page in top_10_geral_list:
        if page not in noticias_usadas:
            df_longo_top10.append([user_id, page, rn])
            noticias_usadas.add(page)
            rn += 1

        if len(df_longo_top10) >= 2:  # Pegar exatamente 2 do Top 10
            break

    # Ao final de cada user, joga o df_longo_top10 dele para o resultado final
    df_longo_top10_final.extend(df_longo_top10)

# Criar DataFrame único com todos os usuários longos
df_longo_cluster = pd.DataFrame(df_longo_cluster, columns=["userId", "page", "rn"])
df_longo_top10_final = pd.DataFrame(df_longo_top10_final, columns=["userId", "page", "rn"])

# Concatenar cluster + top10
df_recomendacoes_longo = pd.concat([df_longo_cluster, df_longo_top10_final])

# Mostrar para validação
print(df_recomendacoes_longo)

# Validação
total_users_longo = df_recomendacoes_longo['userId'].nunique()
total_registros_longo = len(df_recomendacoes_longo)
total_esperado = total_users_longo * 5

print(f"\nQuantidade de usuários longos: {total_users_longo}")
print(f"Quantidade de registros (recomendações): {total_registros_longo}")
print(f"Quantidade esperada (usuários * 5): {total_esperado}")

if total_registros_longo == total_esperado:
    print("Cada usuário longo recebeu exatamente 5 recomendações.")
else:
    print("Problema: Alguns usuários longos receberam menos de 5 recomendações.")


                                                   userId  \
0       000044b36375e7f1a66a9476affc2ddc83c2ec6dd18951...   
1       000044b36375e7f1a66a9476affc2ddc83c2ec6dd18951...   
2       000044b36375e7f1a66a9476affc2ddc83c2ec6dd18951...   
3       00004868f064a8147619ca4d75eac9ccabfbe1169840e6...   
4       00004868f064a8147619ca4d75eac9ccabfbe1169840e6...   
...                                                   ...   
470375  ffff154c70d0fa6bc3c215363df5b77180ebdbf8ffe6a3...   
470376  ffff2c95e9c668ba32f163d1b2573ae67c95ea49e7b380...   
470377  ffff2c95e9c668ba32f163d1b2573ae67c95ea49e7b380...   
470378  ffffee5eea1777ae6686e5286c79e1d3358ff76a73d4ee...   
470379  ffffee5eea1777ae6686e5286c79e1d3358ff76a73d4ee...   

                                                     page  rn  
0                    f6b5d170-48b9-4f8e-88d4-c84b6668f3bd   1  
1                    855d20b7-53f2-4678-a10f-55402d085018   2  
2       esid:conteudo_editorial_g1#materia#https://esp...   3  
3          

In [10]:
# Concatenar todos os DFs
df_recomendacoes = pd.concat([
    df_curto_cluster,
    df_curto_top10_final,
    df_longo_cluster,
    df_longo_top10_final
])

# Ordenar por userId e rn
df_recomendacoes = df_recomendacoes.sort_values(by=["userId", "rn"])

# Mostrar as 10 primeiras linhas
print("\n Top 10 recomendações finais ordenadas:")
display(df_recomendacoes.head(10))



 Top 10 recomendações finais ordenadas:


,userId,page,rn
0,000044b36375e7f1a66a9476affc2ddc83c2ec6dd18951...,f6b5d170-48b9-4f8e-88d4-c84b6668f3bd,1
1,000044b36375e7f1a66a9476affc2ddc83c2ec6dd18951...,855d20b7-53f2-4678-a10f-55402d085018,2
2,000044b36375e7f1a66a9476affc2ddc83c2ec6dd18951...,esid:conteudo_editorial_g1#materia#https://esp...,3
0,000044b36375e7f1a66a9476affc2ddc83c2ec6dd18951...,d2593c3d-2347-40d9-948c-b6065e8459a9,4
1,000044b36375e7f1a66a9476affc2ddc83c2ec6dd18951...,1f32787b-de2b-49be-8c20-ddaeae34cc22,5
3,00004868f064a8147619ca4d75eac9ccabfbe1169840e6...,f6b5d170-48b9-4f8e-88d4-c84b6668f3bd,1
4,00004868f064a8147619ca4d75eac9ccabfbe1169840e6...,855d20b7-53f2-4678-a10f-55402d085018,2
5,00004868f064a8147619ca4d75eac9ccabfbe1169840e6...,esid:conteudo_editorial_g1#materia#https://esp...,3
2,00004868f064a8147619ca4d75eac9ccabfbe1169840e6...,d2593c3d-2347-40d9-948c-b6065e8459a9,4
3,00004868f064a8147619ca4d75eac9ccabfbe1169840e6...,1f32787b-de2b-49be-8c20-ddaeae34cc22,5


In [11]:
# Definir user específico para analisar
user_id = "09dc0ebc8b0c50d81373b975c6328fce9bd0bfc6ebf7314e98dc574aba5cfead"

# Filtrar recomendações do usuário
df_user_recomendacoes = df_recomendacoes[df_recomendacoes["userId"] == user_id].copy()

# Verificar se existe a coluna 'rn'
if 'rn' not in df_user_recomendacoes.columns:
    raise ValueError("A coluna 'rn' não existe no recomendacoes_usuarios.csv. Verifique o processo de geração.")

# Ordenar por rn para garantir a ordem de recomendação
df_user_recomendacoes = df_user_recomendacoes.sort_values(by="rn")

# Fazer join para trazer o cluster e o score da página
df_user_recomendacoes = df_user_recomendacoes.merge(
    df_noticias[["page", "cluster", "score"]],
    on="page",
    how="left"
)

# Mostrar resultado
if df_user_recomendacoes.empty:
    print(f"Nenhuma recomendação encontrada para o user {user_id}.")
else:
    print(f"Recomendações para o user {user_id}:")
    print(df_user_recomendacoes[["userId", "page", "cluster", "score", "rn"]])

    # Obter o cluster recomendado (primeira recomendação)
    cluster_previsto = df_user_recomendacoes.iloc[0]["cluster"]
    print(f"\nCluster previsto para o usuário {user_id}: {cluster_previsto}")



Recomendações para o user 09dc0ebc8b0c50d81373b975c6328fce9bd0bfc6ebf7314e98dc574aba5cfead:
                                              userId  \
0  09dc0ebc8b0c50d81373b975c6328fce9bd0bfc6ebf731...   
1  09dc0ebc8b0c50d81373b975c6328fce9bd0bfc6ebf731...   
2  09dc0ebc8b0c50d81373b975c6328fce9bd0bfc6ebf731...   
3  09dc0ebc8b0c50d81373b975c6328fce9bd0bfc6ebf731...   
4  09dc0ebc8b0c50d81373b975c6328fce9bd0bfc6ebf731...   

                                   page  cluster     score  rn  
0  49f392e2-93ff-4edf-9f54-274a334b7852        6  0.460153   1  
1  3ee8ff4c-2dae-4394-8ff6-abd22556e323        6  0.382050   2  
2  d2bfebf8-2ed7-4e2f-a6a4-b950486a86e7        6  0.376532   3  
3  d2593c3d-2347-40d9-948c-b6065e8459a9        2  0.796550   4  
4  f6b5d170-48b9-4f8e-88d4-c84b6668f3bd        5  0.736898   5  

Cluster previsto para o usuário 09dc0ebc8b0c50d81373b975c6328fce9bd0bfc6ebf7314e98dc574aba5cfead: 6


In [12]:
# Carregar validação e notícias
df_validacao = pd.read_csv("arquivos/validacao.csv", dtype={
    "userId": "string", "page": "string", "timestampHistory": "string"
})
df_validacao["timestampHistory"] = pd.to_datetime(df_validacao["timestampHistory"], errors="coerce")

df_noticias = pd.read_csv("arquivos/noticias_clusterizadas.csv", dtype={
    "page": "string", "cluster": "int", "score": "float"
})

# Obter a primeira página acessada por cada usuário na validação
df_primeira_validacao = (
    df_validacao
    .sort_values(["userId", "timestampHistory"])
    .groupby("userId")
    .head(1)[["userId", "page"]]
)

# Criar uma cópia de df_recomendacoes para validação, preservando o original
df_recomendacoes_validacao = df_recomendacoes.copy()

# Garantir que cada página recomendada na cópia tem seu cluster associado
df_recomendacoes_validacao = df_recomendacoes_validacao.merge(
    df_noticias[["page", "cluster"]],
    on="page",
    how="left"
)

# ---- Validação por página (page) ----

# Juntar recomendações com a primeira página real (usa a cópia)
df_recomendacoes_validacao_page = df_recomendacoes_validacao.merge(
    df_primeira_validacao,
    on="userId",
    suffixes=("_recomendado", "_real")
)

# Marcar acerto exato da página
df_recomendacoes_validacao_page["acertou_page"] = (
    df_recomendacoes_validacao_page["page_recomendado"] == df_recomendacoes_validacao_page["page_real"]
)

# Descobrir a posição (rn) da primeira página correta para cada user
df_acertos_page = (
    df_recomendacoes_validacao_page
    .query("acertou_page")
    .groupby("userId")["rn"]
    .min()
    .reset_index()
    .rename(columns={"rn": "posicao_acerto_page"})
)

# Marcar no dataframe final de páginas qual posição acertou
df_validacao_page_final = (
    df_recomendacoes[["userId", "rn", "page"]]
    .merge(df_acertos_page, on="userId", how="left")
)

df_validacao_page_final["acertou_page"] = (
    df_validacao_page_final["rn"] == df_validacao_page_final["posicao_acerto_page"]
)

# ---- Validação por cluster ----

# Adicionar cluster real da primeira página de cada usuário
df_primeira_validacao = df_primeira_validacao.merge(
    df_noticias[["page", "cluster"]],
    on="page",
    how="left"
)

# Juntar recomendações (com cluster) com o cluster real da primeira página de cada usuário
df_recomendacoes_validacao_cluster = df_recomendacoes_validacao.merge(
    df_primeira_validacao[["userId", "cluster"]],
    on="userId",
    suffixes=("_recomendado", "_real")
)

# Marcar acerto de cluster
df_recomendacoes_validacao_cluster["acertou_cluster"] = (
    df_recomendacoes_validacao_cluster["cluster_recomendado"] == df_recomendacoes_validacao_cluster["cluster_real"]
)

# Descobrir a posição (rn) do primeiro acerto de cluster para cada user
df_acertos_cluster = (
    df_recomendacoes_validacao_cluster
    .query("acertou_cluster")
    .groupby("userId")["rn"]
    .min()
    .reset_index()
    .rename(columns={"rn": "posicao_acerto_cluster"})
)

# Marcar no dataframe final de clusters qual posição acertou
df_validacao_cluster_final = (
    df_recomendacoes_validacao.merge(df_acertos_cluster, on="userId", how="left")
)

df_validacao_cluster_final["acertou_cluster"] = (
    df_validacao_cluster_final["rn"] == df_validacao_cluster_final["posicao_acerto_cluster"]
)

# ---- Cálculo das taxas gerais ----

total_users = df_validacao_page_final["userId"].nunique()

# Taxa de acerto de página
acertos_page = df_validacao_page_final.query("acertou_page")["userId"].nunique()
taxa_acerto_page = acertos_page / total_users

# Taxa de acerto de cluster
acertos_cluster = df_validacao_cluster_final.query("acertou_cluster")["userId"].nunique()
taxa_acerto_cluster = acertos_cluster / total_users

print(f"Total de usuários com recomendações: {total_users}")
print(f"Usuários com acerto de página: {acertos_page}")
print(f"Taxa de acerto de página: {taxa_acerto_page:.2%}")

print(f"Usuários com acerto de cluster: {acertos_cluster}")
print(f"Taxa de acerto de cluster: {taxa_acerto_cluster:.2%}")

# ---- Distribuição de acertos por posição (PAGE) ----

acertos_por_posicao_page = (
    df_validacao_page_final
    .query("acertou_page")
    .groupby("posicao_acerto_page")
    .size()
    .reindex([1, 2, 3, 4, 5], fill_value=0)
)

distribuicao_acertos_page = (acertos_por_posicao_page / total_users * 100).round(2)

print("\nDistribuição de acertos de página por posição:")
for pos, taxa in distribuicao_acertos_page.items():
    print(f"Posição {pos}: {taxa:.2f}%")

# ---- Distribuição de acertos por posição (CLUSTER) ----

acertos_por_posicao_cluster = (
    df_validacao_cluster_final
    .query("acertou_cluster")
    .groupby("posicao_acerto_cluster")
    .size()
    .reindex([1, 2, 3, 4, 5], fill_value=0)
)

distribuicao_acertos_cluster = (acertos_por_posicao_cluster / total_users * 100).round(2)

print("\nDistribuição de acertos de cluster por posição:")
for pos, taxa in distribuicao_acertos_cluster.items():
    print(f"Posição {pos}: {taxa:.2f}%")


Total de usuários com recomendações: 577942
Usuários com acerto de página: 862
Taxa de acerto de página: 0.15%
Usuários com acerto de cluster: 78542
Taxa de acerto de cluster: 13.59%

Distribuição de acertos de página por posição:
Posição 1: 0.02%
Posição 2: 0.01%
Posição 3: 0.11%
Posição 4: 0.01%
Posição 5: 0.01%

Distribuição de acertos de cluster por posição:
Posição 1: 5.75%
Posição 2: 0.57%
Posição 3: 0.75%
Posição 4: 2.67%
Posição 5: 3.86%


In [13]:
# Salvar arquivo 
df_recomendacoes.to_csv("arquivos/recomendacoes_usuarios.csv", index=False)

print("\n Recomendações geradas e salvas em `arquivos/recomendacoes_usuarios.csv`")
print("\n Top 10 notícias gerais salvas em `arquivos/top10_noticias_gerais.csv`")


 Recomendações geradas e salvas em `arquivos/recomendacoes_usuarios.csv`

 Top 10 notícias gerais salvas em `arquivos/top10_noticias_gerais.csv`
